## import thư viện và đọc dữ liệu

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_score,
    recall_score
)
import matplotlib.pyplot as plt
import joblib

RANDOM_STATE = 42
TARGET = "Attack_type"

def find_data_file(filename: str) -> Path:
    # Find the files independently of the notebook working directory.
    cwd = Path.cwd()
    candidates = [
        cwd / filename,
        cwd / "splitted data" / filename,
        cwd / ".." / "splitted data" / filename,
        cwd / ".." / ".." / "splitted data" / filename,
        cwd / "data" / filename,
    ]

    for parent in (cwd, *cwd.parents):
        candidates.append(parent / "splitted data" / filename)
        candidates.append(parent / "data" / "splitted data" / filename)

    for path in candidates:
        path = path.resolve()
        if path.is_file():
            return path

    searched = "\n".join(f"  - {path.resolve()}" for path in candidates)
    raise FileNotFoundError(
        f"Không tìm thấy {filename}. Các đường dẫn đã kiểm tra:\n{searched}"
    )

train_path = find_data_file("train.csv")
validation_path = find_data_file("validation.csv")

train_df = pd.read_csv(train_path)
validation_df = pd.read_csv(validation_path)

print("Train shape:", train_df.shape)
print("Validation shape:", validation_df.shape)
print("Target:", TARGET)
print("Train file:", train_path)
print("Validation file:", validation_path)

## tách x/y và khai báo pipeline

In [ ]:
# Fix: train_df may have been overwritten by the feature-name list.
if TARGET not in train_df.columns:
    train_df = pd.read_csv(train_path)

# If the CSV still does not contain the target, rebuild using the known feature names.
if TARGET not in train_df.columns:
    train_df = pd.read_csv(train_path, names=feature_columns + [TARGET])

X_train = train_df.drop(columns=[TARGET])
y_train = train_df[TARGET]

X_validation = validation_df.drop(columns=[TARGET])
y_validation = validation_df[TARGET]

categorical_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_features = [col for col in X_train.columns if col not in categorical_features]

print("Categorical features:", categorical_features)
print("Number of numeric features:", len(numeric_features))

preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("numeric", "passthrough", numeric_features),
    ],
    remainder="drop"
)

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

model_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", rf_model)
])

model_pipeline

## 3-Fold Cross-Validation trên tập train

In [ ]:
cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=RANDOM_STATE
)

scoring = {
    "accuracy": "accuracy",
    "macro_f1": "f1_macro",
    "weighted_f1": "f1_weighted",
    "macro_precision": "precision_macro",
    "macro_recall": "recall_macro"
}

cv_results = cross_validate(
    estimator=model_pipeline,
    X=X_train,
    y=y_train,
    cv=cv,
    scoring=scoring,
    return_train_score=False,
    n_jobs=1
)

cv_results_df = pd.DataFrame({
    "Fold": [1, 2, 3],
    "accuracy": cv_results["test_accuracy"],
    "macro_f1": cv_results["test_macro_f1"],
    "weighted_f1": cv_results["test_weighted_f1"],
    "macro_precision": cv_results["test_macro_precision"],
    "macro_recall": cv_results["test_macro_recall"]
})

display(cv_results_df)

summary = cv_results_df.drop(columns="Fold").agg(["mean", "std"]).T
summary.columns = ["Mean", "Std"]
print("3-Fold CV summary:")
display(summary)

## Huấn luyện trên toàn bộ tập train và đánh giá validation

In [ ]:
model_pipeline.fit(X_train, y_train)

y_pred = model_pipeline.predict(X_validation)

validation_metrics = {
    "accuracy": accuracy_score(y_validation, y_pred),
    "macro_f1": f1_score(y_validation, y_pred, average="macro", zero_division=0),
    "weighted_f1": f1_score(y_validation, y_pred, average="weighted", zero_division=0),
    "macro_precision": precision_score(y_validation, y_pred, average="macro", zero_division=0),
    "macro_recall": recall_score(y_validation, y_pred, average="macro", zero_division=0),
}
print("Validation metrics:")
for metric_name, metric_value in validation_metrics.items():
    print(f"{metric_name}: {metric_value:.6f}")
print("\\nClassification report:")
print(classification_report(y_validation, y_pred, zero_division=0))

In [ ]:
# Confusion matrix trên validation
labels = model_pipeline.named_steps["model"].classes_
cm = confusion_matrix(y_validation, y_pred, labels=labels)

fig, ax = plt.subplots(figsize=(12, 10))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(ax=ax, xticks_rotation=90, cmap="Blues", colorbar=False)
plt.title("Confusion Matrix - Validation")
plt.tight_layout()
plt.show()

In [ ]:
from pathlib import Path
import numpy as np

RESULT_DIR = Path.cwd().resolve()
RESULT_DIR.mkdir(parents=True, exist_ok=True)

cv_results_df.to_csv(RESULT_DIR / "E05_cv_folds.csv", index=False, encoding="utf-8-sig")
summary.to_csv(RESULT_DIR / "E05_cv_summary.csv", index=True, index_label="metric", encoding="utf-8-sig")
pd.DataFrame([validation_metrics]).to_csv(
    RESULT_DIR / "E05_validation_metrics.csv", index=False, encoding="utf-8-sig"
)
confusion = pd.DataFrame(cm, index=labels, columns=labels)
confusion.index.name = "Actual"
confusion.to_csv(RESULT_DIR / "E05_validation_confusion_matrix.csv", encoding="utf-8-sig")

support = cm.sum(axis=1)
predicted_support = cm.sum(axis=0)
true_positive = np.diag(cm)
precision = np.divide(true_positive, predicted_support, out=np.zeros_like(true_positive, dtype=float), where=predicted_support != 0)
recall = np.divide(true_positive, support, out=np.zeros_like(true_positive, dtype=float), where=support != 0)
f1 = np.divide(2 * precision * recall, precision + recall, out=np.zeros_like(precision), where=(precision + recall) != 0)
per_class = pd.DataFrame({"precision": precision, "recall": recall, "f1_score": f1, "support": support}, index=labels)
per_class.index.name = "class"
per_class.to_csv(RESULT_DIR / "E05_validation_per_class.csv", encoding="utf-8-sig")
fig.savefig(RESULT_DIR / "E05_validation_confusion_matrix.png", dpi=180, bbox_inches="tight")
